In [0]:
# NOTEBOOK   : silver_orders
# PURPOSE    : Clean and transform bronze.orders table
# SOURCE     : fincompliance_catalog.bronze.orders
# DESTINATION: fincompliance_catalog.silver.orders
# TRANSFORMATIONS:
#   - Deduplicate on order_id
#   - Cast 5 string date columns to TimestampType
#   - Extract order_year, order_month, order_day_of_week
#   - Calculate delivery_delay_days
#   - Classify delivery as on_time or late
#   - Handle nulls in delivery date columns
#   - Add order_processing_days
#   - Drop bronze audit columns
#   - Add silver_updated_at audit column

In [0]:
%run ../configs/config

In [0]:
authenticate_spark(spark)
set_catalog(spark)

In [0]:
from pyspark.sql.functions import (
    col,                # refers to a column by name
    to_timestamp,       # converts string to timestamp
    year,               # extracts year from timestamp
    month,              # extracts month from timestamp
    dayofweek,          # extracts day of week from timestamp
    datediff,           # calculates difference between dates
    when,               # if/else conditions
    lit,                # converts python value to spark column
    current_timestamp,  # gets current date and time
    round as spark_round # rounds decimal numbers
)

In [0]:
# Read bronze.orders table
df_orders = spark.table(f"{BRONZE_DB}.orders")

# Show basic information
print("=" * 60)
print("BRONZE.ORDERS — Before Transformation")
print("=" * 60)
print(f"Total rows    : {df_orders.count():,}")
print(f"Total columns : {len(df_orders.columns)}")

# Show all column names and their data types
print("\nColumn names and data types:")
for field in df_orders.schema.fields:
    print(f"  → {field.name:<40} : {field.dataType}")

# Show sample data before transformation
print("\nSample data BEFORE transformation (3 rows):")
df_orders.show(3, truncate=True)

In [0]:
# Check nulls in all date columns
print("Null counts in date columns:")


date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for c in date_columns:
    null_count = df_orders.filter(col(c).isNull()).count()
    print(f" {c:<40}:{null_count:,} nulls")



In [0]:
# Check order status breakdown
print("Order status breakdown:")
df_orders.groupBy("order_status") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

In [0]:
# DEDUPLICATION
# Remove duplicate rows based on order_id
# order_id is the primary key — must be unique
# Count rows BEFORE deduplication
df_orders = spark.table(f"{BRONZE_DB}.orders")

rows_before = df_orders.count()

print(f"Rows BEFORE deduplication : {rows_before:,}")

# Apply deduplication
df_orders_silver = df_orders.dropDuplicates(["order_id"])

# Count rows AFTER deduplication
rows_after = df_orders_silver.count()
print(f"Rows AFTER deduplication  : {rows_after:,}")

# Show how many duplicates were removed
duplicates_removed = rows_before - rows_after
print(f"Duplicates removed        : {duplicates_removed:,}")

if duplicates_removed == 0:
    print("No duplicates found")
else:
    print(f"{duplicates_removed:,} duplicates removed")

In [0]:
# EXTRACT DATE DIMENSIONS
# Extract year, month, day of week from purchase timestamp
# These columns DO NOT exist in bronze
# Power BI needs these for time based analysis

# Show BEFORE adding date dimensions
print("BEFORE adding date dimensions:")
df_orders_silver.select(
    "order_id",
    "order_purchase_timestamp"
).show(5, truncate=True)

# Extract date dimensions
df_orders_silver = df_orders_silver \
    .withColumn(
        "order_year",
        year(col("order_purchase_timestamp"))
    ) \
    .withColumn(
        "order_month",
        month(col("order_purchase_timestamp"))
    ) \
    .withColumn(
        "order_day_of_week",
        dayofweek(col("order_purchase_timestamp"))
    )

# Show AFTER adding date dimensions
print("\nAFTER adding date dimensions:")
df_orders_silver.select(
    "order_id",
    "order_purchase_timestamp",
    "order_year",
    "order_month",
    "order_day_of_week"
).show(5, truncate=True)

# Show year distribution
print("\nOrders by year:")
df_orders_silver \
    .groupBy("order_year") \
    .count() \
    .orderBy("order_year") \
    .show()

# Show month distribution
print("Orders by month:")
df_orders_silver \
    .groupBy("order_month") \
    .count() \
    .orderBy("order_month") \
    .show()

# Show day of week distribution
print("Orders by day of week:")
print("(1=Sunday 2=Monday 3=Tuesday 4=Wednesday")
print(" 5=Thursday 6=Friday 7=Saturday)")
df_orders_silver \
    .groupBy("order_day_of_week") \
    .count() \
    .orderBy("order_day_of_week") \
    .show()

print("Date dimensions extracted")



In [0]:

# CALCULATE DELIVERY DELAY DAYS
# Calculate how many days early or late each order arrived
# Business metric to be calculated
# Positive value = delivered LATE
# Negative value = delivered EARLY
# Zero = delivered EXACTLY on time
# NULL = not delivered yet


# Show the two date columns we are working with
print("Date columns we are calculating from:")
df_orders_silver.select(
    "order_id",
    "order_status",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
).show(5, truncate=True)

# Calculate delivery_delay_days
df_orders_silver = df_orders_silver.withColumn(
    "delivery_delay_days",
    when(
        col("order_delivered_customer_date").isNull(),
        None
    )
    .otherwise(
        datediff(
            col("order_delivered_customer_date"),
            col("order_estimated_delivery_date")
        )
    )
)

# Show results
print("\nDelivery delay days sample:")
df_orders_silver.select(
    "order_id",
    "order_status",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_delay_days"
).show(10, truncate=True)

# Show statistics
print("\nDelivery delay statistics:")
df_orders_silver \
    .filter(col("delivery_delay_days").isNotNull()) \
    .select("delivery_delay_days") \
    .summary("min", "max", "mean", "count") \
    .show()

# Show breakdown
print("\nDelivery performance breakdown:")
on_time = df_orders_silver \
    .filter(col("delivery_delay_days") <= 0) \
    .count()
late = df_orders_silver \
    .filter(col("delivery_delay_days") > 0) \
    .count()
not_delivered = df_orders_silver \
    .filter(col("delivery_delay_days").isNull()) \
    .count()

total = df_orders_silver.count()

print(f"  On time / Early : {on_time:>8,} orders "
      f"({on_time/total*100:.1f}%)")
print(f"  Late            : {late:>8,} orders "
      f"({late/total*100:.1f}%)")
print(f"  Not delivered   : {not_delivered:>8,} orders "
      f"({not_delivered/total*100:.1f}%)")
print(f"  Total           : {total:>8,} orders")

print("\n Delivery delay days calculated")

In [0]:
# CLASSIFY DELIVERY STATUS
# Classify each order as on_time, late or not_delivered
# Based on delivery_delay_days calculated in Cell 8
# This column DID NOT EXIST in bronze

# Show BEFORE adding delivery status
print("BEFORE adding delivery status:")
df_orders_silver.select(
    "order_id",
    "order_status",
    "delivery_delay_days"
).show(5, truncate=True)

# Apply delivery status classification
df_orders_silver = df_orders_silver.withColumn(
    "delivery_status",
    when(
        col("order_delivered_customer_date").isNull(),
        "not_delivered"
    )
    .when(
        col("delivery_delay_days") <= 0,
        "on_time"
    )
    .when(
        col("delivery_delay_days") > 0,
        "late"
    )
    .otherwise("unknown")
)

# Show AFTER adding delivery status
print("\nAFTER adding delivery status:")
df_orders_silver.select(
    "order_id",
    "order_status",
    "delivery_delay_days",
    "delivery_status"
).show(10, truncate=True)

# Show delivery status breakdown
print("\nDelivery status breakdown:")
df_orders_silver \
    .groupBy("delivery_status") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

# Cross check with order_status
print("\nDelivery status vs order status:")
df_orders_silver \
    .groupBy("order_status", "delivery_status") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(20, truncate=False)

print("Delivery status classification complete")



In [0]:
# CALCULATE ORDER PROCESSING DAYS
# How many days between purchase and approval?
# This measures how fast Olist processes orders
# This column DID NOT EXIST in bronze

# Show the two columns we are working with
print("Columns we are calculating from:")
df_orders_silver.select(
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at"
).show(5, truncate=True)

# Calculate order processing days
df_orders_silver = df_orders_silver.withColumn(
    "order_processing_days",
    when(
        col("order_approved_at").isNull(),
        None
    )
    .otherwise(
        datediff(
            col("order_approved_at"),
            col("order_purchase_timestamp")
        )
    )
)

# Show results
print("\nOrder processing days sample:")
df_orders_silver.select(
    "order_id",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_processing_days"
).show(10, truncate=True)

# Show statistics
print("\nOrder processing days statistics:")
df_orders_silver \
    .filter(col("order_processing_days").isNotNull()) \
    .select("order_processing_days") \
    .summary("min", "max", "mean", "count") \
    .show()

# Show processing days breakdown
print("\nProcessing days breakdown:")
df_orders_silver \
    .filter(col("order_processing_days").isNotNull()) \
    .groupBy("order_processing_days") \
    .count() \
    .orderBy("order_processing_days") \
    .show(10, truncate=False)

print("Order processing days calculated")


In [0]:
# CELL 11 - HANDLE NULLS
# Document and handle null values in all columns
# We already handled nulls in derived columns
# Now we handle nulls in original date columns

# Show null counts for ALL columns
print("=" * 60)
print("NULL COUNTS — ALL COLUMNS")
print("=" * 60)

for c in df_orders_silver.columns:
    null_count = df_orders_silver \
        .filter(col(c).isNull()) \
        .count()
    if null_count == 0:
        print(f"{c:<40} : {null_count:,} nulls")
    else:
        print(f"{c:<40} : {null_count:,} nulls")


# Handle nulls in order_approved_at
# 160 orders were never approved
# We keep as null — cannot fill with fake date
# But we add a flag column to identify them

print("\nHandling nulls in order_approved_at:")
print("→ Keeping as null (order was never approved)")
print("→ Adding is_approved flag column")

df_orders_silver = df_orders_silver.withColumn(
    "is_approved",
    when(col("order_approved_at").isNull(), 0)
    .otherwise(1)
)

approved_count = df_orders_silver \
    .filter(col("is_approved") == 1).count()
not_approved_count = df_orders_silver \
    .filter(col("is_approved") == 0).count()

print(f"  Approved orders     : {approved_count:,}")
print(f"  Not approved orders : {not_approved_count:,}")


# Handle nulls in order_delivered_carrier_date
# 1,783 orders never handed to carrier
# Keep as null — adding flag column

print("\nHandling nulls in order_delivered_carrier_date:")
print("→ Keeping as null (order never handed to carrier)")
print("→ Adding is_shipped flag column")

df_orders_silver = df_orders_silver.withColumn(
    "is_shipped",
    when(col("order_delivered_carrier_date").isNull(), 0)
    .otherwise(1)
)

shipped_count = df_orders_silver \
    .filter(col("is_shipped") == 1).count()
not_shipped_count = df_orders_silver \
    .filter(col("is_shipped") == 0).count()

print(f"  Shipped orders     : {shipped_count:,}")
print(f"  Not shipped orders : {not_shipped_count:,}")

# Handle nulls in order_delivered_customer_date
# 2,965 orders never delivered to customer
# Keep as null — delivery_status already handles this

print("\nHandling nulls in order_delivered_customer_date:")
print("→ Keeping as null (order never delivered)")
print("→ delivery_status column already classifies these")
print("  as not_delivered")

delivered_count = df_orders_silver \
    .filter(
        col("order_delivered_customer_date").isNotNull()
    ).count()
not_delivered_count = df_orders_silver \
    .filter(
        col("order_delivered_customer_date").isNull()
    ).count()

print(f"  Delivered orders     : {delivered_count:,}")
print(f"  Not delivered orders : {not_delivered_count:,}")

print("Null handling complete")



In [0]:
# CELL 12 - DROP BRONZE AUDIT COLUMNS
# Remove ingestion_timestamp and source_file
# These belong to bronze layer only
# Silver has its own audit column

# Show columns BEFORE dropping
print("Columns BEFORE dropping audit columns:")
print(f"Total columns : {len(df_orders_silver.columns)}")
for col_name in df_orders_silver.columns:
    print(f"  → {col_name}")

# Drop bronze audit columns
df_orders_silver = df_orders_silver.drop(
    "ingestion_timestamp",
    "source_file"
)

# Show columns AFTER dropping
print(f"\nColumns AFTER dropping audit columns:")
print(f"Total columns : {len(df_orders_silver.columns)}")
for col_name in df_orders_silver.columns:
    print(f"  → {col_name}")

print("Bronze audit columns dropped")
    

In [0]:
# ADD SILVER AUDIT COLUMN
# Add silver_updated_at to track when this
# silver transformation ran
# Standard practice in every production pipeline

# Add silver audit column
df_orders_silver = df_orders_silver.withColumn(
    "silver_updated_at",
    current_timestamp()
)

# Show final column list
print("Final columns in silver.orders:")
print(f"Total columns : {len(df_orders_silver.columns)}")
for col_name in df_orders_silver.columns:
    print(f"  → {col_name}")

# Show final row count
print(f"Total rows : {df_orders_silver.count():,}")

# Show sample of final DataFrame
print("\nSample data (3 rows):")
df_orders_silver.select(
    "order_id",
    "order_status",
    "order_year",
    "order_month",
    "delivery_delay_days",
    "delivery_status",
    "order_processing_days",
    "is_approved",
    "is_shipped",
    "silver_updated_at"
).show(3, truncate=True)

print("Silver audit column added")

In [0]:
# WRITE TO SILVER
# Write the transformed DataFrame to
# fincompliance_catalog.silver.orders
# as a Delta table

print("Writing to silver.orders...")
print(f"Destination : {SILVER_DB}.orders")
print(f"Format      : Delta")
print(f"Mode        : Overwrite")
print(f"Total rows  : {df_orders_silver.count():,}")
print(f"Total cols  : {len(df_orders_silver.columns)}")

# Write transformed DataFrame to silver layer
(
    df_orders_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_DB}.orders")
)

print("\nSuccessfully written to silver.orders")
print(f"Table : {SILVER_DB}.orders")

In [0]:
# CELL 15 - SILVER ORDERS VERIFICATION
print("=" * 60)
print("SILVER.ORDERS VERIFICATION")
print("=" * 60)

df_silver = spark.table(f"{SILVER_DB}.orders")

# CHECK 1 — Row count
bronze_count = spark.table(f"{BRONZE_DB}.orders").count()
silver_count = df_silver.count()
print(f"\nCHECK 1 — Row counts:")
print(f"  Bronze : {bronze_count:,}")
print(f"  Silver : {silver_count:,}")
if bronze_count == silver_count:
    print(f"Row counts match")
else:
    print(f"Difference : {bronze_count - silver_count:,}")

# CHECK 2 — New columns exist
print(f"\nCHECK 2 — New columns verification:")
new_columns = [
    "order_year",
    "order_month",
    "order_day_of_week",
    "delivery_delay_days",
    "delivery_status",
    "order_processing_days",
    "is_approved",
    "is_shipped",
    "silver_updated_at"
]
for c in new_columns:
    if c in df_silver.columns:
        print(f" {c:<30} exists")
    else:
        print(f"{c:<30} MISSING")

# CHECK 3 — Delivery status breakdown
print(f"\nCHECK 3 — Delivery status breakdown:")
df_silver \
    .groupBy("delivery_status") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

# CHECK 4 — Year distribution
print(f"CHECK 4 — Year distribution:")
df_silver \
    .groupBy("order_year") \
    .count() \
    .orderBy("order_year") \
    .show(truncate=False)

print("=" * 60)
print("silver.orders verification complete!")
print(f"Table : {SILVER_DB}.orders")
print(f"Rows  : {silver_count:,}")
print(f"Cols  : {len(df_silver.columns)}")
print("=" * 60)
